# Frozen DINOv3 + Logistic Regression Baseline for BRISC Brain Tumor MRI Classification

This notebook provides a rigorous frozen DINOv3 linear-probe baseline for the BRISC brain tumor MRI dataset.  
The entire DINOv3 backbone is frozen; only a sklearn `LogisticRegression` head is trained on the extracted CLS-token features.  
No layers are unfrozen, no augmentation is applied, and the official test set is never used for tuning.

In [ ]:
%pip install -q transformers timm huggingface_hub imagehash Pillow

## Section 1 — Setup and Authentication

The Hugging Face token is retrieved securely via Colab Secrets (`userdata`) or an environment variable.  
No token is ever printed or hardcoded.

In [ ]:
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HuggingFace authentication successful.")
else:
    print("No HF_TOKEN found. Public models will still work.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile

zip_path = "/content/drive/MyDrive/Data/BT_Images.zip"
if os.path.exists(zip_path) and not os.path.exists("/content/train"):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall("/content")
    print("Done.")
else:
    print("Dataset already available or zip not found at expected path.")

## Section 2 — Central Configuration

In [ ]:
from pathlib import Path

CONFIG = {
    "model_id": "facebook/dinov3-vitb16-pretrain-lvd1689m",
    "img_size": 224,
    "batch_size": 32,
    "val_split": 0.15,
    "seeds": [42, 123, 2026],
    "n_folds": 5,
    "C": 1.0,
    "C_candidates": [0.01, 0.1, 1.0, 10.0, 100.0],
    "max_iter": 2000,
    "output_dir": "baseline_results",
    "train_root": "/content/train",
    "test_root": "/content/test",
}

OUT = Path(CONFIG["output_dir"])
OUT.mkdir(parents=True, exist_ok=True)

for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Section 3 — Reproducibility

In [ ]:
import sys
import random
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
from tqdm.auto import tqdm

from transformers import AutoModel
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score, classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score, roc_auc_score,
)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env_info = {
    "python": sys.version,
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": __import__("transformers").__version__,
    "sklearn": sklearn.__version__,
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else "N/A",
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    "device": str(device),
    "timestamp": datetime.now().isoformat(),
}

env_path = OUT / "environment.txt"
with open(env_path, "w") as f:
    for k, v in env_info.items():
        f.write(f"{k}: {v}\n")

for k, v in env_info.items():
    print(f"  {k}: {v}")
print(f"\nSaved to {env_path}")

## Section 4 — Dataset Audit

In [ ]:
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def scan_directory(root: str, split_name: str) -> pd.DataFrame:
    rows = []
    root_path = Path(root)
    for class_dir in sorted(root_path.iterdir()):
        if not class_dir.is_dir():
            continue
        for img_path in sorted(class_dir.iterdir()):
            if img_path.suffix.lower() not in VALID_EXT:
                continue
            try:
                img = Image.open(img_path)
                w, h = img.size
            except Exception:
                w, h = -1, -1
            rows.append({
                "filepath": str(img_path),
                "filename": img_path.name,
                "class": class_dir.name,
                "split": split_name,
                "width": w,
                "height": h,
            })
    return pd.DataFrame(rows)


df_train_audit = scan_directory(CONFIG["train_root"], "train")
df_test_audit = scan_directory(CONFIG["test_root"], "test")
df_audit = pd.concat([df_train_audit, df_test_audit], ignore_index=True)

print(f"Train images: {len(df_train_audit)}")
print(f"Test images:  {len(df_test_audit)}")
print(f"Total:        {len(df_audit)}")

print("\nTrain class distribution:")
print(df_train_audit["class"].value_counts().sort_index().to_string())

print("\nTest class distribution:")
print(df_test_audit["class"].value_counts().sort_index().to_string())

print("\nImage dimensions (width x height):")
print(f"  Min:  {df_audit['width'].min()} x {df_audit['height'].min()}")
print(f"  Max:  {df_audit['width'].max()} x {df_audit['height'].max()}")
print(f"  Mode: {df_audit['width'].mode().iloc[0]} x {df_audit['height'].mode().iloc[0]}")

train_classes = sorted(df_train_audit["class"].unique())
test_classes = sorted(df_test_audit["class"].unique())
assert train_classes == test_classes, (
    f"Class mismatch: train={train_classes}, test={test_classes}"
)
print(f"\nClass names verified identical: {train_classes}")

df_audit.to_csv(OUT / "dataset_summary.csv", index=False)
print(f"Saved to {OUT / 'dataset_summary.csv'}")

## Section 5 — Exact Duplicate Check (SHA-256)

In [ ]:
def sha256_file(filepath: str) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


print("Computing SHA-256 hashes...")
df_audit["sha256"] = [sha256_file(fp) for fp in tqdm(df_audit["filepath"], desc="Hashing")]

dup_within_train = df_audit[df_audit["split"] == "train"][df_audit["sha256"].duplicated(keep=False)]
dup_within_test = df_audit[df_audit["split"] == "test"][df_audit["sha256"].duplicated(keep=False)]

train_hashes = set(df_audit.loc[df_audit["split"] == "train", "sha256"])
test_hashes = set(df_audit.loc[df_audit["split"] == "test", "sha256"])
cross_dup_hashes = train_hashes & test_hashes
dup_cross = df_audit[df_audit["sha256"].isin(cross_dup_hashes)]

all_dups = pd.concat([
    dup_within_train.assign(dup_type="within_train"),
    dup_within_test.assign(dup_type="within_test"),
    dup_cross.assign(dup_type="cross_train_test"),
], ignore_index=True).drop_duplicates(subset=["filepath", "dup_type"])

all_dups.to_csv(OUT / "exact_duplicates.csv", index=False)

print(f"\nDuplicates within train: {len(dup_within_train)}")
print(f"Duplicates within test:  {len(dup_within_test)}")
print(f"Exact duplicates across train/test: {len(cross_dup_hashes)} unique hashes, {len(dup_cross)} images")

if len(cross_dup_hashes) > 0:
    print("WARNING: Exact duplicates exist between train and test sets.")
else:
    print("No exact duplicates found between train and test sets.")

print(f"Saved to {OUT / 'exact_duplicates.csv'}")

## Section 6 — Near-Duplicate Check (Perceptual Hashing)

In [ ]:
import imagehash

HAMMING_THRESHOLD = 5

print(f"Computing perceptual hashes (pHash, Hamming threshold = {HAMMING_THRESHOLD})...")


def compute_phash(filepath: str) -> str:
    try:
        return str(imagehash.phash(Image.open(filepath)))
    except Exception:
        return None


df_audit["phash"] = [compute_phash(fp) for fp in tqdm(df_audit["filepath"], desc="pHash")]

train_mask = df_audit["split"] == "train"
test_mask = df_audit["split"] == "test"

train_phashes = df_audit.loc[train_mask, ["filepath", "class", "phash"]].reset_index(drop=True)
test_phashes = df_audit.loc[test_mask, ["filepath", "class", "phash"]].reset_index(drop=True)

near_dups = []
for i, t_row in tqdm(test_phashes.iterrows(), total=len(test_phashes), desc="Comparing"):
    if t_row["phash"] is None:
        continue
    t_hash = imagehash.hex_to_hash(t_row["phash"])
    for j, tr_row in train_phashes.iterrows():
        if tr_row["phash"] is None:
            continue
        dist = t_hash - imagehash.hex_to_hash(tr_row["phash"])
        if dist <= HAMMING_THRESHOLD:
            near_dups.append({
                "test_filepath": t_row["filepath"],
                "test_class": t_row["class"],
                "train_filepath": tr_row["filepath"],
                "train_class": tr_row["class"],
                "hamming_distance": dist,
            })

df_near_dups = pd.DataFrame(near_dups)
df_near_dups.to_csv(OUT / "near_duplicates.csv", index=False)

print(f"\nNear-duplicate pairs (Hamming <= {HAMMING_THRESHOLD}): {len(df_near_dups)}")
if len(df_near_dups) > 0:
    print("WARNING: Potential near-duplicates found between train and test.")
    print(df_near_dups.head(10).to_string(index=False))
else:
    print("No near-duplicates found between train and test sets.")

print(f"Saved to {OUT / 'near_duplicates.csv'}")

## Section 7 — Stratified Train / Validation Split

The official BRISC test set is never touched.  
The training set is split into train and validation partitions using stratified sampling.

In [ ]:
train_filepaths = df_train_audit["filepath"].tolist()
train_labels_str = df_train_audit["class"].tolist()

CLASS_NAMES = sorted(set(train_labels_str))
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

train_labels_int = [CLASS_TO_IDX[c] for c in train_labels_str]

idx_train, idx_val = train_test_split(
    list(range(len(train_filepaths))),
    test_size=CONFIG["val_split"],
    stratify=train_labels_int,
    random_state=42,
)

manifest_rows = []
for i in idx_train:
    manifest_rows.append({"filepath": train_filepaths[i], "class": train_labels_str[i], "partition": "train"})
for i in idx_val:
    manifest_rows.append({"filepath": train_filepaths[i], "class": train_labels_str[i], "partition": "validation"})
for _, row in df_test_audit.iterrows():
    manifest_rows.append({"filepath": row["filepath"], "class": row["class"], "partition": "test"})

df_manifest = pd.DataFrame(manifest_rows)
df_manifest.to_csv(OUT / "split_manifest.csv", index=False)

print("Partition sizes:")
print(df_manifest["partition"].value_counts().to_string())

print("\nClass counts per partition:")
print(df_manifest.groupby(["partition", "class"]).size().unstack(fill_value=0).to_string())

print("\nReliable patient identifiers are not available in the current dataset metadata, "
      "therefore patient-level independence cannot be independently verified.")

print(f"\nSaved to {OUT / 'split_manifest.csv'}")

## Section 8 — Preprocessing

**No MRI-altering augmentation is used in this baseline.**  
Images are resized to 224 x 224, converted to tensors, and normalized with ImageNet statistics.  
No horizontal/vertical flips, rotation, color jitter, RandAugment, random erasing, or any other augmentation is applied.  
This ensures the baseline measures pure frozen-backbone representational quality without confounds from data augmentation.

In [ ]:
transform = T.Compose([
    T.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class ImageListDataset(Dataset):
    """Dataset from a list of (filepath, label) pairs."""

    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]


print(f"Preprocessing: Resize({CONFIG['img_size']}x{CONFIG['img_size']}) -> ToTensor -> ImageNet Normalize")
print("No augmentation applied.")

## Section 9 — DINOv3 Feature Extraction

In [ ]:
print(f"Loading backbone: {CONFIG['model_id']}")

backbone = AutoModel.from_pretrained(CONFIG["model_id"], token=HF_TOKEN)
backbone = backbone.to(device)
backbone.eval()

for param in backbone.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in backbone.parameters())
trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
pct_trainable = 100.0 * trainable_params / total_params if total_params > 0 else 0.0

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Percentage trainable: {pct_trainable:.4f}%")
assert trainable_params == 0, f"Expected 0 trainable params, got {trainable_params}"

hidden_dim = getattr(backbone.config, "hidden_size", 768)
print(f"Feature dimension:    {hidden_dim}")
assert hidden_dim == 768, f"Expected 768, got {hidden_dim}"

In [ ]:
@torch.no_grad()
def extract_features(filepaths, labels, backbone, transform, batch_size, device, desc=""):
    ds = ImageListDataset(filepaths, labels, transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    all_feats, all_labs = [], []
    for images, labs in tqdm(loader, desc=desc):
        images = images.to(device)
        out = backbone(images)
        feats = out.last_hidden_state[:, 0]
        all_feats.append(feats.cpu().numpy())
        all_labs.append(labs.numpy())
    return np.concatenate(all_feats), np.concatenate(all_labs)


all_train_fps = [train_filepaths[i] for i in range(len(train_filepaths))]
all_train_labs = [train_labels_int[i] for i in range(len(train_labels_int))]

test_fps = df_test_audit["filepath"].tolist()
test_labs = [CLASS_TO_IDX[c] for c in df_test_audit["class"].tolist()]

print("Extracting features for ALL training images (used for train+val splits)...")
X_all_train, y_all_train = extract_features(
    all_train_fps, all_train_labs, backbone, transform, CONFIG["batch_size"], device, "Train+Val"
)

print("Extracting features for test images...")
X_test, y_test = extract_features(
    test_fps, test_labs, backbone, transform, CONFIG["batch_size"], device, "Test"
)

X_train_split = X_all_train[idx_train]
y_train_split = y_all_train[idx_train]
X_val_split = X_all_train[idx_val]
y_val_split = y_all_train[idx_val]

np.savez(OUT / "train_features.npz", features=X_train_split, labels=y_train_split,
         filepaths=np.array([train_filepaths[i] for i in idx_train]))
np.savez(OUT / "val_features.npz", features=X_val_split, labels=y_val_split,
         filepaths=np.array([train_filepaths[i] for i in idx_val]))
np.savez(OUT / "test_features.npz", features=X_test, labels=y_test,
         filepaths=np.array(test_fps))

print(f"\nTrain features: {X_train_split.shape}")
print(f"Val features:   {X_val_split.shape}")
print(f"Test features:  {X_test.shape}")
print(f"Cached to {OUT}")

## Section 10 — Primary Four-Class Baseline

In [ ]:
def compute_multiclass_metrics(y_true, y_pred, class_names):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def compute_per_class_metrics(y_true, y_pred, class_names):
    rows = []
    for i, cls in enumerate(class_names):
        mask = y_true == i
        rows.append({
            "class": cls,
            "precision": precision_score(y_true == i, y_pred == i, zero_division=0),
            "recall": recall_score(y_true == i, y_pred == i, zero_division=0),
            "f1": f1_score(y_true == i, y_pred == i, zero_division=0),
            "support": int(mask.sum()),
        })
    return pd.DataFrame(rows)


def make_lr_pipeline(C=1.0, random_state=42):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=C,
            solver="lbfgs",
            max_iter=CONFIG["max_iter"],
            random_state=random_state,
        )),
    ])


def select_C(X_train, y_train, X_val, y_val, random_state=42):
    best_C, best_score = CONFIG["C"], -1.0
    for C in CONFIG["C_candidates"]:
        pipe = make_lr_pipeline(C=C, random_state=random_state)
        pipe.fit(X_train, y_train)
        score = balanced_accuracy_score(y_val, pipe.predict(X_val))
        if score > best_score:
            best_score, best_C = score, C
    return best_C


best_C = select_C(X_train_split, y_train_split, X_val_split, y_val_split, random_state=42)
CONFIG["C"] = best_C
print(f"Selected C={best_C} via validation balanced accuracy (with StandardScaler)")

clf_val = make_lr_pipeline(C=best_C, random_state=42)
clf_val.fit(X_train_split, y_train_split)

val_preds = clf_val.predict(X_val_split)
val_metrics = compute_multiclass_metrics(y_val_split, val_preds, CLASS_NAMES)
print("Validation metrics:")
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}")

clf = make_lr_pipeline(C=best_C, random_state=42)
clf.fit(X_all_train, y_all_train)

test_preds = clf.predict(X_test)
test_metrics = compute_multiclass_metrics(y_test, test_preds, CLASS_NAMES)
print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

per_class = compute_per_class_metrics(y_test, test_preds, CLASS_NAMES)
print("\nPer-class test metrics:")
print(per_class.to_string(index=False))

pd.DataFrame([test_metrics]).to_csv(OUT / "multiclass_test_metrics.csv", index=False)
per_class.to_csv(OUT / "multiclass_per_class_metrics.csv", index=False)
print(f"\nSaved to {OUT}")

## Section 11 — Confusion Matrices

In [ ]:
def plot_confusion_matrix(cm, class_names, title, save_prefix, normalize=False):
    if normalize:
        cm_plot = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fmt_func = lambda v: f"{v:.1%}"
    else:
        cm_plot = cm.astype(float)
        fmt_func = lambda v: f"{int(v)}"

    fig, ax = plt.subplots(figsize=(7, 6))
    cmap = LinearSegmentedColormap.from_list("blues", ["#ffffff", "#3274a1"])
    im = ax.imshow(cm_plot, interpolation="nearest", cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    n = len(class_names)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=10)
    ax.set_yticklabels(class_names, fontsize=10)

    thresh = cm_plot.max() / 2.0
    for i in range(n):
        for j in range(n):
            color = "white" if cm_plot[i, j] > thresh else "black"
            ax.text(j, i, fmt_func(cm_plot[i, j]), ha="center", va="center",
                    color=color, fontsize=12, fontweight="bold")

    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("Actual", fontsize=12)
    ax.set_title(title, fontsize=13, fontweight="bold")
    plt.tight_layout()

    for ext in ["png", "pdf"]:
        fig.savefig(f"{save_prefix}.{ext}", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


cm = confusion_matrix(y_test, test_preds)

plot_confusion_matrix(cm, CLASS_NAMES,
    "Four-Class Confusion Matrix (Raw Counts)",
    str(OUT / "multiclass_confusion_matrix"), normalize=False)

plot_confusion_matrix(cm, CLASS_NAMES,
    "Four-Class Confusion Matrix (Row-Normalized)",
    str(OUT / "multiclass_confusion_matrix_normalized"), normalize=True)

## Section 12 — Three-Seed Experiment

Three independent stratified train/validation splits are created using seeds 42, 123, and 2026.  
DINOv3 features are **not** re-extracted — they are deterministic for frozen backbones with identical preprocessing.  
Each Logistic Regression is evaluated on the same untouched official test set.

In [ ]:
seed_results = []

for seed in CONFIG["seeds"]:
    idx_tr, idx_va = train_test_split(
        list(range(len(all_train_fps))),
        test_size=CONFIG["val_split"],
        stratify=all_train_labs,
        random_state=seed,
    )

    X_tr = X_all_train[idx_tr]
    y_tr = y_all_train[idx_tr]
    X_va = X_all_train[idx_va]
    y_va = y_all_train[idx_va]

    set_seed(seed)
    seed_C = select_C(X_tr, y_tr, X_va, y_va, random_state=seed)
    c = make_lr_pipeline(C=seed_C, random_state=seed)
    c.fit(X_tr, y_tr)
    preds = c.predict(X_test)
    m = compute_multiclass_metrics(y_test, preds, CLASS_NAMES)
    m["seed"] = seed
    seed_results.append(m)
    print(f"Seed {seed}: Acc={m['accuracy']:.4f}  BalAcc={m['balanced_accuracy']:.4f}  "
          f"MacroF1={m['macro_f1']:.4f}  WeightedF1={m['weighted_f1']:.4f}  Kappa={m['kappa']:.4f}")

df_seeds = pd.DataFrame(seed_results)
cols = [c for c in df_seeds.columns if c != "seed"]
df_seeds.to_csv(OUT / "three_seed_results.csv", index=False)

summary_rows = []
for col in cols:
    vals = df_seeds[col].values
    m, s = vals.mean(), vals.std(ddof=1)
    ci95 = 1.96 * s / np.sqrt(len(vals))
    summary_rows.append({
        "metric": col, "mean": m, "std": s,
        "ci95_lower": m - ci95, "ci95_upper": m + ci95,
    })

df_seed_summary = pd.DataFrame(summary_rows)
df_seed_summary.to_csv(OUT / "three_seed_summary.csv", index=False)

print("\nPaper-friendly results:")
for _, row in df_seed_summary.iterrows():
    print(f"  {row['metric']}: {row['mean']:.4f} +/- {row['std']:.4f}  "
          f"[95% CI: {row['ci95_lower']:.4f} - {row['ci95_upper']:.4f}]")

## Section 13 — Five-Fold Cross-Validation

Five-fold cross-validation was conducted **only** on the official BRISC training partition.  
The official test set remained untouched.

In [ ]:
skf = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=42)

fold_results = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all_train, y_all_train), 1):
    X_tr, y_tr = X_all_train[tr_idx], y_all_train[tr_idx]
    X_va, y_va = X_all_train[va_idx], y_all_train[va_idx]

    fold_C = select_C(X_tr, y_tr, X_va, y_va, random_state=42)
    c = make_lr_pipeline(C=fold_C, random_state=42)
    c.fit(X_tr, y_tr)
    preds = c.predict(X_va)
    m = compute_multiclass_metrics(y_va, preds, CLASS_NAMES)
    m["fold"] = fold
    fold_results.append(m)
    print(f"Fold {fold}: Acc={m['accuracy']:.4f}  BalAcc={m['balanced_accuracy']:.4f}  "
          f"MacroF1={m['macro_f1']:.4f}  WeightedF1={m['weighted_f1']:.4f}")

df_folds = pd.DataFrame(fold_results)
metric_cols = [c for c in df_folds.columns if c != "fold"]
df_folds.to_csv(OUT / "cv_fold_results.csv", index=False)

cv_summary_rows = []
for col in metric_cols:
    vals = df_folds[col].values
    m, s = vals.mean(), vals.std(ddof=1)
    ci95 = 1.96 * s / np.sqrt(len(vals))
    cv_summary_rows.append({
        "metric": col, "mean": m, "std": s,
        "ci95_lower": m - ci95, "ci95_upper": m + ci95,
    })

df_cv_summary = pd.DataFrame(cv_summary_rows)
df_cv_summary.to_csv(OUT / "cv_summary.csv", index=False)

print("\nFive-Fold CV Summary:")
for _, row in df_cv_summary.iterrows():
    print(f"  {row['metric']}: {row['mean']:.4f} +/- {row['std']:.4f}  "
          f"[95% CI: {row['ci95_lower']:.4f} - {row['ci95_upper']:.4f}]")

## Section 14 — Independent Binary Baseline

In [ ]:
BINARY_MAP = {}
for cls in CLASS_NAMES:
    BINARY_MAP[CLASS_TO_IDX[cls]] = 0 if cls == "no_tumor" else 1

BINARY_NAMES = ["no_tumor", "tumor"]

y_train_bin = np.array([BINARY_MAP[y] for y in y_train_split])
y_val_bin = np.array([BINARY_MAP[y] for y in y_val_split])
y_all_train_bin = np.array([BINARY_MAP[y] for y in y_all_train])
y_test_bin = np.array([BINARY_MAP[y] for y in y_test])

best_C_bin = select_C(X_train_split, y_train_bin, X_val_split, y_val_bin, random_state=42)
clf_bin = make_lr_pipeline(C=best_C_bin, random_state=42)
clf_bin.fit(X_all_train, y_all_train_bin)

bin_preds = clf_bin.predict(X_test)
bin_probs = clf_bin.predict_proba(X_test)[:, 1]

print("A separate binary Logistic Regression classifier was trained directly using binary labels. "
      "Binary predictions were not produced by regrouping multiclass predictions.")

tn = ((bin_preds == 0) & (y_test_bin == 0)).sum()
fp = ((bin_preds == 1) & (y_test_bin == 0)).sum()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

bin_metrics = {
    "accuracy": accuracy_score(y_test_bin, bin_preds),
    "balanced_accuracy": balanced_accuracy_score(y_test_bin, bin_preds),
    "precision": precision_score(y_test_bin, bin_preds, zero_division=0),
    "recall": recall_score(y_test_bin, bin_preds, zero_division=0),
    "specificity": specificity,
    "f1": f1_score(y_test_bin, bin_preds, zero_division=0),
    "roc_auc": roc_auc_score(y_test_bin, bin_probs),
    "pr_auc": average_precision_score(y_test_bin, bin_probs),
    "kappa": cohen_kappa_score(y_test_bin, bin_preds),
}

print("\nIndependent Binary Classifier — Test Metrics:")
for k, v in bin_metrics.items():
    print(f"  {k}: {v:.4f}")

pd.DataFrame([bin_metrics]).to_csv(OUT / "binary_test_metrics.csv", index=False)

# --- Collapsed multiclass detection performance ---
collapsed_preds = np.array([BINARY_MAP[p] for p in test_preds])

tn_c = ((collapsed_preds == 0) & (y_test_bin == 0)).sum()
fp_c = ((collapsed_preds == 1) & (y_test_bin == 0)).sum()
spec_c = tn_c / (tn_c + fp_c) if (tn_c + fp_c) > 0 else 0.0

collapsed_metrics = {
    "accuracy": accuracy_score(y_test_bin, collapsed_preds),
    "balanced_accuracy": balanced_accuracy_score(y_test_bin, collapsed_preds),
    "precision": precision_score(y_test_bin, collapsed_preds, zero_division=0),
    "recall": recall_score(y_test_bin, collapsed_preds, zero_division=0),
    "specificity": spec_c,
    "f1": f1_score(y_test_bin, collapsed_preds, zero_division=0),
    "kappa": cohen_kappa_score(y_test_bin, collapsed_preds),
}

print("\nCollapsed Multiclass Detection Performance:")
for k, v in collapsed_metrics.items():
    print(f"  {k}: {v:.4f}")

comparison = pd.DataFrame([
    {"method": "independent_binary_classifier", **bin_metrics},
    {"method": "collapsed_multiclass", **collapsed_metrics},
])
comparison.to_csv(OUT / "binary_vs_collapsed.csv", index=False)
print(f"\nSaved to {OUT / 'binary_vs_collapsed.csv'}")

In [ ]:
# Binary confusion matrix
cm_bin = confusion_matrix(y_test_bin, bin_preds)
plot_confusion_matrix(cm_bin, BINARY_NAMES,
    "Binary Confusion Matrix (Independent Classifier)",
    str(OUT / "binary_confusion_matrix"), normalize=False)

# ROC curve
fpr, tpr, _ = roc_curve(y_test_bin, bin_probs)
roc_auc_val = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="#3274a1", lw=2, label=f"ROC (AUC = {roc_auc_val:.4f})")
ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("Binary ROC Curve", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
for ext in ["png", "pdf"]:
    fig.savefig(OUT / f"binary_roc_curve.{ext}", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# Precision-Recall curve
prec_vals, rec_vals, _ = precision_recall_curve(y_test_bin, bin_probs)
pr_auc_val = auc(rec_vals, prec_vals)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(rec_vals, prec_vals, color="#e1812c", lw=2, label=f"PR (AUC = {pr_auc_val:.4f})")
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Binary Precision-Recall Curve", fontsize=13, fontweight="bold")
ax.legend(loc="lower left", fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
for ext in ["png", "pdf"]:
    fig.savefig(OUT / f"binary_pr_curve.{ext}", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

## Section 15 — Bootstrap Confidence Intervals

Image-level bootstrap confidence intervals (2000 samples, seed 42).  
Reliable patient identifiers are not available in the current dataset metadata, therefore these are image-level bootstrap confidence intervals.

In [ ]:
N_BOOTSTRAP = 2000
rng = np.random.RandomState(42)


def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=N_BOOTSTRAP, ci=0.95):
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        try:
            scores.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception:
            continue
    scores = np.array(scores)
    alpha = (1 - ci) / 2
    return np.percentile(scores, 100 * alpha), np.percentile(scores, 100 * (1 - alpha))


def bootstrap_ci_proba(y_true, y_proba, metric_fn, n_boot=N_BOOTSTRAP, ci=0.95):
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            scores.append(metric_fn(y_true[idx], y_proba[idx]))
        except Exception:
            continue
    scores = np.array(scores)
    alpha = (1 - ci) / 2
    return np.percentile(scores, 100 * alpha), np.percentile(scores, 100 * (1 - alpha))


print("Computing image-level bootstrap confidence intervals (n=2000)...")

boot_rows = []

# Multiclass
for name, fn in [
    ("multiclass_accuracy", accuracy_score),
    ("multiclass_balanced_accuracy", balanced_accuracy_score),
    ("multiclass_macro_f1", lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0)),
    ("multiclass_weighted_f1", lambda yt, yp: f1_score(yt, yp, average="weighted", zero_division=0)),
]:
    lo, hi = bootstrap_ci(y_test, test_preds, fn)
    pt = fn(y_test, test_preds)
    boot_rows.append({"metric": name, "point_estimate": pt, "ci95_lower": lo, "ci95_upper": hi})
    print(f"  {name}: {pt:.4f} [{lo:.4f} - {hi:.4f}]")

# Binary
def specificity_fn(yt, yp):
    tn_ = ((yp == 0) & (yt == 0)).sum()
    fp_ = ((yp == 1) & (yt == 0)).sum()
    return tn_ / (tn_ + fp_) if (tn_ + fp_) > 0 else 0.0

for name, fn in [
    ("binary_accuracy", accuracy_score),
    ("binary_recall", lambda yt, yp: recall_score(yt, yp, zero_division=0)),
    ("binary_specificity", specificity_fn),
    ("binary_f1", lambda yt, yp: f1_score(yt, yp, zero_division=0)),
]:
    lo, hi = bootstrap_ci(y_test_bin, bin_preds, fn)
    pt = fn(y_test_bin, bin_preds)
    boot_rows.append({"metric": name, "point_estimate": pt, "ci95_lower": lo, "ci95_upper": hi})
    print(f"  {name}: {pt:.4f} [{lo:.4f} - {hi:.4f}]")

for name, fn in [
    ("binary_roc_auc", roc_auc_score),
    ("binary_pr_auc", average_precision_score),
]:
    lo, hi = bootstrap_ci_proba(y_test_bin, bin_probs, fn)
    pt = fn(y_test_bin, bin_probs)
    boot_rows.append({"metric": name, "point_estimate": pt, "ci95_lower": lo, "ci95_upper": hi})
    print(f"  {name}: {pt:.4f} [{lo:.4f} - {hi:.4f}]")

df_boot = pd.DataFrame(boot_rows)
df_boot.to_csv(OUT / "bootstrap_confidence_intervals.csv", index=False)
print(f"\nSaved to {OUT / 'bootstrap_confidence_intervals.csv'}")

## Section 16 — Final Baseline Summary Table

In [ ]:
def fmt_mean_std(df_summary, metric):
    row = df_summary[df_summary["metric"] == metric].iloc[0]
    return f"{row['mean']:.4f} +/- {row['std']:.4f}"

def fmt_ci(df_summary, metric):
    row = df_summary[df_summary["metric"] == metric].iloc[0]
    return f"[{row['ci95_lower']:.4f} - {row['ci95_upper']:.4f}]"

summary_table = pd.DataFrame([
    {
        "experiment": "Frozen DINOv3 + LR (4-class test)",
        "accuracy": f"{test_metrics['accuracy']:.4f}",
        "balanced_accuracy": f"{test_metrics['balanced_accuracy']:.4f}",
        "macro_precision": f"{test_metrics['macro_precision']:.4f}",
        "macro_recall": f"{test_metrics['macro_recall']:.4f}",
        "macro_f1": f"{test_metrics['macro_f1']:.4f}",
        "weighted_f1": f"{test_metrics['weighted_f1']:.4f}",
        "kappa": f"{test_metrics['kappa']:.4f}",
    },
    {
        "experiment": "Three-seed mean +/- SD",
        "accuracy": fmt_mean_std(df_seed_summary, "accuracy"),
        "balanced_accuracy": fmt_mean_std(df_seed_summary, "balanced_accuracy"),
        "macro_precision": fmt_mean_std(df_seed_summary, "macro_precision"),
        "macro_recall": fmt_mean_std(df_seed_summary, "macro_recall"),
        "macro_f1": fmt_mean_std(df_seed_summary, "macro_f1"),
        "weighted_f1": fmt_mean_std(df_seed_summary, "weighted_f1"),
        "kappa": fmt_mean_std(df_seed_summary, "kappa"),
    },
    {
        "experiment": "Five-fold CV mean +/- SD",
        "accuracy": fmt_mean_std(df_cv_summary, "accuracy"),
        "balanced_accuracy": fmt_mean_std(df_cv_summary, "balanced_accuracy"),
        "macro_precision": fmt_mean_std(df_cv_summary, "macro_precision"),
        "macro_recall": fmt_mean_std(df_cv_summary, "macro_recall"),
        "macro_f1": fmt_mean_std(df_cv_summary, "macro_f1"),
        "weighted_f1": fmt_mean_std(df_cv_summary, "weighted_f1"),
        "kappa": fmt_mean_std(df_cv_summary, "kappa"),
    },
    {
        "experiment": "Independent binary DINOv3 + LR",
        "accuracy": f"{bin_metrics['accuracy']:.4f}",
        "balanced_accuracy": f"{bin_metrics['balanced_accuracy']:.4f}",
        "macro_precision": f"{bin_metrics['precision']:.4f}",
        "macro_recall": f"{bin_metrics['recall']:.4f}",
        "macro_f1": f"{bin_metrics['f1']:.4f}",
        "weighted_f1": f"{bin_metrics['f1']:.4f}",
        "kappa": f"{bin_metrics['kappa']:.4f}",
    },
])

summary_table.to_csv(OUT / "baseline_summary.csv", index=False)

print("=" * 120)
print("FINAL BASELINE SUMMARY")
print("=" * 120)
print(summary_table.to_string(index=False))
print(f"\nSaved to {OUT / 'baseline_summary.csv'}")

## Section 17 — Final Publication Figures

In [ ]:
# Cross-validation fold accuracy bar chart
fig, ax = plt.subplots(figsize=(7, 4))
folds = df_folds["fold"].values
accs = df_folds["accuracy"].values
mean_acc = accs.mean()
ax.bar(folds, accs, color="#3274a1", edgecolor="black", linewidth=0.5)
ax.axhline(mean_acc, color="#e1812c", linestyle="--", lw=1.5, label=f"Mean = {mean_acc:.4f}")
ax.set_xlabel("Fold", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Five-Fold Cross-Validation Accuracy", fontsize=13, fontweight="bold")
ax.set_xticks(folds)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
for ext in ["png", "pdf"]:
    fig.savefig(OUT / f"cv_fold_accuracy.{ext}", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# Three-seed accuracy bar chart
fig, ax = plt.subplots(figsize=(6, 4))
seeds = df_seeds["seed"].values
seed_accs = df_seeds["accuracy"].values
mean_seed = seed_accs.mean()
ax.bar(range(len(seeds)), seed_accs, color="#3274a1", edgecolor="black", linewidth=0.5)
ax.axhline(mean_seed, color="#e1812c", linestyle="--", lw=1.5, label=f"Mean = {mean_seed:.4f}")
ax.set_xlabel("Seed", fontsize=12)
ax.set_ylabel("Test Accuracy", fontsize=12)
ax.set_title("Three-Seed Test Accuracy", fontsize=13, fontweight="bold")
ax.set_xticks(range(len(seeds)))
ax.set_xticklabels([str(s) for s in seeds])
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
for ext in ["png", "pdf"]:
    fig.savefig(OUT / f"three_seed_accuracy.{ext}", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print("All figures saved as PNG and PDF.")

## Section 18 — Machine-Readable Experiment Manifest

In [ ]:
output_files = sorted([str(p) for p in OUT.iterdir()])

manifest = {
    "dataset_name": "BRISC Brain Tumor MRI",
    "model_id": CONFIG["model_id"],
    "feature_type": "CLS token (last_hidden_state[:, 0])",
    "feature_dimension": hidden_dim,
    "classifier": "StandardScaler + sklearn.linear_model.LogisticRegression",
    "hyperparameters": {
        "C": CONFIG["C"],
        "C_candidates": CONFIG["C_candidates"],
        "feature_scaling": "StandardScaler (fit on train split only for C selection)",
        "solver": "lbfgs",
        "max_iter": CONFIG["max_iter"],
    },
    "class_names": CLASS_NAMES,
    "num_classes": NUM_CLASSES,
    "seeds": CONFIG["seeds"],
    "n_folds": CONFIG["n_folds"],
    "preprocessing": {
        "resize": f"{CONFIG['img_size']}x{CONFIG['img_size']}",
        "normalization": "ImageNet (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])",
        "augmentation": "None",
    },
    "num_images": {
        "train_total": len(df_train_audit),
        "test_total": len(df_test_audit),
    },
    "software_versions": env_info,
    "output_files": output_files,
}

manifest_path = OUT / "experiment_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"Experiment manifest saved to {manifest_path}")
print(json.dumps(manifest, indent=2, default=str))

## Section 19 — Reviewer Concerns Addressed by This Baseline

In [ ]:
reviewer_table = pd.DataFrame([
    {
        "Reviewer Concern": "Missing baseline",
        "How Addressed": "Frozen DINOv3 linear probe (LogisticRegression on CLS features)",
        "Output": "multiclass_test_metrics.csv, baseline_summary.csv",
    },
    {
        "Reviewer Concern": "Single seed",
        "How Addressed": "Three independent stratified runs (seeds 42, 123, 2026) with mean, SD, and 95% CI",
        "Output": "three_seed_results.csv, three_seed_summary.csv",
    },
    {
        "Reviewer Concern": "Under-reported cross-validation",
        "How Addressed": "Five-fold stratified CV on training partition only, with per-fold, mean, SD, and 95% CI",
        "Output": "cv_fold_results.csv, cv_summary.csv",
    },
    {
        "Reviewer Concern": "Binary experiment ambiguity",
        "How Addressed": "Separate binary classifier trained on binary labels, plus collapsed multiclass comparison",
        "Output": "binary_test_metrics.csv, binary_vs_collapsed.csv",
    },
    {
        "Reviewer Concern": "MRI augmentation concern",
        "How Addressed": "Minimal preprocessing — resize, tensor, ImageNet normalization only. No MRI-altering augmentation",
        "Output": "experiment_manifest.json (preprocessing section)",
    },
    {
        "Reviewer Concern": "Missing statistical rigor",
        "How Addressed": "SD, 95% CI, and 2000-sample bootstrap confidence intervals for all key metrics",
        "Output": "bootstrap_confidence_intervals.csv, three_seed_summary.csv, cv_summary.csv",
    },
    {
        "Reviewer Concern": "Incomplete metrics",
        "How Addressed": "Balanced accuracy, macro/weighted precision/recall/F1, Cohen kappa, ROC AUC, PR AUC",
        "Output": "multiclass_test_metrics.csv, binary_test_metrics.csv",
    },
    {
        "Reviewer Concern": "Data leakage concern",
        "How Addressed": "SHA-256 exact duplicate check and perceptual hash near-duplicate check across train/test",
        "Output": "exact_duplicates.csv, near_duplicates.csv",
    },
    {
        "Reviewer Concern": "Reproducibility",
        "How Addressed": "Saved split manifest, environment versions, full experiment manifest, and cached features",
        "Output": "split_manifest.csv, environment.txt, experiment_manifest.json",
    },
])

print("=" * 120)
print("REVIEWER CONCERNS ADDRESSED BY THIS BASELINE")
print("=" * 120)
print(reviewer_table.to_string(index=False))
print("\nBaseline notebook complete.")